# How unusual is the 2026 El Nino, so far?

The Oceanic Nino Index (ONI) is a three-month running mean of sea surface temperature
anomalies in the Nino 3.4 region of the equatorial Pacific (5N-5S, 120W-170W). Above
+0.5 C signals El Nino, below -0.5 C signals La Nina.

This notebook loads NOAA CPC's full ONI table from 1950 to the present, checks how the
current year compares against every prior year, and plots the result.

**Two things worth knowing before you read anything into the numbers.**

1. As of 1 February 2026, per NWS Public Information Statement 26-05, CPC's *official*
   monitoring index is no longer ONI but **RONI** (Relative Oceanic Nino Index), which
   subtracts the tropical mean SST anomaly (20N-20S) before rescaling. The reason is that
   the tropical ocean as a whole has warmed, so a raw Nino 3.4 anomaly increasingly picks
   up background warming rather than ENSO itself. This notebook uses ONI because it has a
   consistent long record for the historical comparison, but the two indices answer
   different questions and give different numbers.
2. The most recent ONI values are provisional and can be revised for up to two months.

Source: https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/oni/v6/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

## 1. Load the data

`oni_v6.csv` is a direct transcription of the CPC ONI v6 table: one row per year, one
column per overlapping three-month season (DJF, JFM, FMA ... NDJ).

Point `CSV_URL` at wherever you have put the file. To refresh it later, copy the current
table off the CPC page linked above.

In [ ]:
CSV_URL = "oni_v6.csv" # make sure it is in the same folder as the notebook

SEASONS = ["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ",
           "JJA", "JAS", "ASO", "SON", "OND", "NDJ"]

df = pd.read_csv(CSV_URL)
print(f"{len(df)} years, {df['year'].min()} to {df['year'].max()}")
df.tail(4)

## 2. Reshape to a continuous series

The wide table is convenient to read but awkward to analyse. Melting it into one long
series lets us slide a window across the whole record, including across year boundaries.

In [ ]:
ts = (df.melt(id_vars="year", value_vars=SEASONS,
              var_name="season", value_name="oni")
        .dropna(subset=["oni"]))

ts["sidx"] = ts["season"].map({s: i for i, s in enumerate(SEASONS)})
ts = ts.sort_values(["year", "sidx"]).reset_index(drop=True)

latest = ts.iloc[-1]
print(f"{len(ts)} seasons on record. Most recent: "
      f"{latest['season']} {int(latest['year'])} = {latest['oni']:+.1f}")

## 3. Question one: is this the warmest the Pacific has been at this point in the year?

Comparing the current season against the same season in every prior year. This controls
for the seasonal cycle, since El Nino events typically develop in spring and peak around
the turn of the year, so a mid-year value is not comparable to a December one.

In [ ]:
cur_season = latest["season"]
cur_year = int(latest["year"])

same_season = (ts[ts["season"] == cur_season]
               .sort_values("oni", ascending=False)
               .reset_index(drop=True))

rank = same_season.index[same_season["year"] == cur_year][0] + 1
print(f"{cur_season} {cur_year} ranks #{rank} of {len(same_season)} "
      f"years on record for that season.\n")

print(f"Warmest {cur_season} seasons since 1950:")
for _, r in same_season.head(6).iterrows():
    peak = df.loc[df["year"] == r["year"], "NDJ"].values[0]
    peak_txt = f"peaked at {peak:+.1f} by NDJ" if pd.notna(peak) else "still developing"
    print(f"  {int(r['year'])}: {r['oni']:+.1f}   ({peak_txt})")

## 4. Question two: how fast did it get here?

Sliding a five-step window across the full continuous series to find the largest rises
over any five overlapping seasons since 1950.

In [ ]:
WINDOW = 5
ts["rise"] = ts["oni"] - ts["oni"].shift(WINDOW)
ts["from_season"] = ts["season"].shift(WINDOW)
ts["from_year"] = ts["year"].shift(WINDOW)
ts["from_oni"] = ts["oni"].shift(WINDOW)

top = ts.dropna(subset=["rise"]).nlargest(8, "rise")
print(f"Largest rises over {WINDOW} overlapping seasons, 1950-present:\n")
for _, r in top.iterrows():
    print(f"  {r['from_season']} {int(r['from_year'])} {r['from_oni']:+.1f}"
          f"  ->  {r['season']} {int(r['year'])} {r['oni']:+.1f}"
          f"   change {r['rise']:+.1f}")

## 5. The chart

Every year drawn as its own line across the twelve overlapping seasons. Grey for context,
colour for the two closest historical analogues and the current year.

Change `HIGHLIGHT` to compare against different years.

In [ ]:
CURRENT = cur_year
HIGHLIGHT = {1997: "#e07a2f", 2015: "#3e7fb0"}
CURRENT_COLOR = "#c0392b"
x = np.arange(len(SEASONS))

fig, ax = plt.subplots(figsize=(9, 9.6), dpi=120)
fig.subplots_adjust(top=0.855, bottom=0.115, left=0.105, right=0.945)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# every other year, as faint context
for _, r in df.iterrows():
    if int(r["year"]) in HIGHLIGHT or int(r["year"]) == CURRENT:
        continue
    ax.plot(x, r[SEASONS].astype(float).values,
            color="#cdcdcd", lw=0.8, alpha=0.6, zorder=1)

# historical analogues
for yr, colour in HIGHLIGHT.items():
    y = df.loc[df["year"] == yr, SEASONS].astype(float).values[0]
    ax.plot(x, y, color=colour, lw=2.4, zorder=3)
    ax.annotate(str(yr), (x[-1], y[-1]), xytext=(7, 0), textcoords="offset points",
                color=colour, fontsize=12, fontweight="bold", va="center")

# current year, stopping at the last reported season
y_cur = df.loc[df["year"] == CURRENT, SEASONS].astype(float).values[0]
mask = ~np.isnan(y_cur)
last_i = int(np.where(mask)[0][-1])
ax.plot(x[mask], y_cur[mask], color=CURRENT_COLOR, lw=4.2,
        zorder=5, solid_capstyle="round")
ax.scatter([last_i], [y_cur[last_i]], s=110, color=CURRENT_COLOR, zorder=6)
ax.annotate(str(CURRENT), (last_i, y_cur[last_i]), xytext=(12, 8),
            textcoords="offset points", color=CURRENT_COLOR,
            fontsize=15, fontweight="bold")

# reference lines
ax.axhline(0, color="#555", lw=0.9, zorder=2)
for level, label in [(0.5, "El Nino threshold"), (1.5, "Strong"), (2.0, "Very strong")]:
    ax.axhline(level, color="#a5a5a5", lw=0.7, ls=(0, (4, 4)), zorder=1)
    ax.annotate(label, (0.985, level), xycoords=("axes fraction", "data"),
                ha="right", xytext=(0, 5), textcoords="offset points",
                fontsize=9, color="#8a8a8a")

ax.set_xticks(x)
ax.set_xticklabels(SEASONS, fontsize=10)
ax.set_ylim(-2.2, 2.9)
ax.set_xlim(-0.4, len(SEASONS) + 0.4)
ax.set_ylabel("Oceanic Nino Index   (3-month mean, Nino 3.4 region)",
              fontsize=10.5, color="#444")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color("#c5c5c5")
ax.spines["bottom"].set_color("#c5c5c5")
ax.tick_params(colors="#666")

fig.text(0.105, 0.955, "The Pacific has never been this warm this early",
         fontsize=19, fontweight="bold", color="#1a1a1a", ha="left")
fig.text(0.105, 0.918,
         "Each grey line is one year since 1950, traced season by season.\n"
         f"By {cur_season.replace('MJJ', 'May-July')}, {CURRENT} sits above every year on record.",
         fontsize=11.5, color="#555", ha="left", va="top")
# fig.text(0.105, 0.038,
#          "Data: NOAA Climate Prediction Center, Oceanic Nino Index v6 (ERSSTv6), "
#          f"1950 through {cur_season} {CURRENT}.\n"
#          "The most recent values are provisional. RONI, not ONI, is now the official "
#          "index CPC uses for monitoring.",
#          fontsize=8.5, color="#777", ha="left", va="top")

plt.savefig("oni_trajectories.png", facecolor="white")
plt.show()

## Notes and caveats

- Being the warmest year on record *at this point in the year* is not a forecast of where
  it ends up. 1997 and 2015 both went on to become the largest events in the record, but
  two prior cases is not a sample, and the two lines here are selected precisely because
  they are the closest analogues.
- CPC's own guidance is that even the strongest events do not produce the expected
  impacts everywhere. Stronger events tilt the odds; they do not guarantee outcomes.
- CPC's forecast probabilities are stated in **RONI**, not the ONI plotted here. The two
  are not interchangeable and the thresholds do not map onto each other one for one.
- ONI v6 uses ERSSTv6. Earlier published ONI values were computed on ERSSTv5, so the
  historical numbers here differ slightly from older versions of the same chart.
- CPC updates the ONI table by the 5th of each month, and issues an ENSO Diagnostic
  Discussion on the second Thursday.

**Sources**

- ONI table: https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/oni/v6/
- RONI: https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/roni/
- ENSO Diagnostic Discussion: https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso_advisory/ensodisc.shtml